# Исследование извлечения таблиц из нормативных документов (ML-подход)

Цель этого ноутбука — проверить, можно ли извлекать **структуру таблиц** (строки/колонки и их содержимое) из произвольного PDF с помощью готовых ML-моделей, а не regex/`pdfplumber.extract_tables`, которые завязаны на конкретный документ и разваливаются на разорванных по страницам таблицах.

Проверяем на пилотном СП 22.13330.2016 (text-based PDF, есть текстовый слой), но сам подход не зависит от документа: это layout-детекция по растру + распознавание структуры таблицы по растру.

## Контекст: почему regex — не панацея

Текущий парсер (`normatives_pipeline.parser.extractors`) строит таблицы через `pdfplumber` + эвристики (префикс `табл.`, поиск подписи `Т а б л и ц а N`, перенос номера на следующую страницу). Ограничения:

- подпись может быть на предыдущей странице или с разрывом (колонтитулы);
- разорванные по страницам таблицы требуют ручной склейки;
- всё завязано на русские слова `табл.`/`Таблица` и формат номера `N.N`;
- не распознаётся семантика (заголовок vs данные, объединённые ячейки).

ML-модели, обученные на разметке документов, должны обобщать лучше.

## Подход (три слоя)

1. **Layout-детекция** — `DocLayout-YOLO` (YOLOv10, обучен на DocStructBench): находит блоки `table`, `table_caption`, `isolate_formula`, `title` и т.д. на растровой странице.
2. **Структура таблицы** — `TableTransformer` (Microsoft, PubTables-1M): по вырезанному изображению таблицы находит `table row` / `table column` / `table column header`.
3. **Содержимое ячеек** — для text-based PDF берём текст из `pdfplumber` по bbox ячейки; для сканов подключаем OCR (следующий этап).

Обе модели работают по изображению, поэтому не зависят от текстового слоя и от языка.

In [ ]:
from __future__ import annotations

import sys
import warnings

warnings.filterwarnings("ignore")

from pathlib import Path

import cv2
import numpy as np
import pdfplumber
import torch
from transformers import AutoImageProcessor, TableTransformerForObjectDetection

from doclayout_yolo import YOLOv10
from huggingface_hub import hf_hub_download

# Путь к корню репозитория (надёжно при запуске из любого cwd).
ROOT = Path.cwd().resolve()
while not (ROOT / "docs" / "normative-base").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "ml" / "src"))

PDF = ROOT / "docs" / "normative-base" / "СП 22.13330.2016.pdf"
PAGES_DIR = ROOT / "ml" / "research" / "pages"  # 400-DPI PNG из research.ipynb

torch.set_grad_enabled(False)
print("ROOT:", ROOT)
print("cuda:", torch.cuda.is_available())


## Слой 1: layout-детекция DocLayout-YOLO

Загружаем веса и смотрим, какие классы блоков модель находит на пилотных страницах. Классы DocStructBench: `title`, `plain text`, `abandon`, `figure`, `figure_caption`, `table`, `table_caption`, `table_footnote`, `isolate_formula`, `formula_caption`.

In [ ]:
LAYOUT_WEIGHTS = hf_hub_download(
    "juliozhao/DocLayout-YOLO-DocStructBench",
    "doclayout_yolo_docstructbench_imgsz1024.pt",
)
layout_model = YOLOv10(LAYOUT_WEIGHTS)

PILOT_PAGES = ["015", "028", "038", "054"]


def layout_classes(page_no: str) -> dict[str, int]:
    img = cv2.imread(str(PAGES_DIR / f"page_{page_no}.png"))
    result = list(layout_model.predict(img, conf=0.2, imgsz=1024, verbose=False))[0]
    counts: dict[str, int] = {}
    for cls in result.boxes.cls:
        name = result.names[int(cls)]
        counts[name] = counts.get(name, 0) + 1
    return counts


for p in PILOT_PAGES:
    print(p, dict(sorted(layout_classes(p).items(), key=lambda kv: -kv[1])))


**Вывод:** модель уверенно отделяет `table`, `table_caption`, `isolate_formula`, `title` на растровой странице — это уже даёт независимое от языка выделение таблиц, в отличие от regex-поиска подписи `Таблица`.

## Слой 2: детекция bbox таблицы

Берём bbox блока класса `table` из DocLayout-YOLO (в координатах 400-DPI растра).

In [ ]:
def detect_table_bbox(page_no: str) -> tuple[int, int, int, int] | None:
    img = cv2.imread(str(PAGES_DIR / f"page_{page_no}.png"))
    for result in layout_model.predict(img, conf=0.2, imgsz=1024, verbose=False):
        for box, cls in zip(result.boxes.xyxy, result.boxes.cls):
            if result.names[int(cls)] == "table":
                return tuple(int(v) for v in box)
    return None


detect_table_bbox("015")


## Слой 2 (продолжение): структура таблицы TableTransformer

Вырезаем таблицу и прогоняем `table-transformer-structure-recognition`. Модель возвращает bbox строк (`table row`) и колонок (`table column`, `table column header`). По ним строим сетку ячеек.

In [ ]:
STRUCT_PROCESSOR = AutoImageProcessor.from_pretrained(
    "microsoft/table-transformer-structure-recognition"
)
STRUCT_MODEL = TableTransformerForObjectDetection.from_pretrained(
    "microsoft/table-transformer-structure-recognition", low_cpu_mem_usage=False
).eval()


def table_grid(page_no: str) -> tuple[list[tuple[int, int]], list[tuple[int, int]]]:
    img = cv2.imread(str(PAGES_DIR / f"page_{page_no}.png"))
    x1, y1, x2, y2 = detect_table_bbox(page_no)
    crop = cv2.cvtColor(img[y1:y2, x1:x2], cv2.COLOR_BGR2RGB)
    h, w = crop.shape[:2]
    inputs = STRUCT_PROCESSOR(images=crop, return_tensors="pt")
    with torch.no_grad():
        out = STRUCT_MODEL(**inputs)
    res = STRUCT_PROCESSOR.post_process_object_detection(
        out, threshold=0.7, target_sizes=[(h, w)]
    )[0]
    rows: list[tuple[int, int]] = []
    cols: list[tuple[int, int]] = []
    for label, box in zip(res["labels"], res["boxes"]):
        name = STRUCT_MODEL.config.id2label[label.item()]
        xa, ya, xb, yb = box.tolist()
        if name == "table row":
            rows.append((round(ya), round(yb)))
        elif name in ("table column", "table column header"):
            cols.append((round(xa), round(xb)))
    return sorted(set(rows)), sorted(set(cols))


table_grid("015")


## Слой 3: содержимое ячеек

Для text-based PDF координаты растра переводятся в PDF points (`px / (dpi/72)`), и текст каждой ячейки берётся через `pdfplumber.within_bbox`. Для сканов здесь же подключается OCR (TODO).

In [ ]:
def reconstruct_table(page_no: str) -> list[list[str]]:
    x1, y1, x2, y2 = detect_table_bbox(page_no)
    rows, cols = table_grid(page_no)
    with pdfplumber.open(PDF) as pdf:
        page = pdf.pages[int(page_no) - 1]
        img = cv2.imread(str(PAGES_DIR / f"page_{page_no}.png"))
        sx = img.shape[1] / page.width
        sy = img.shape[0] / page.height
        out: list[list[str]] = []
        for ra, rb in rows:
            line: list[str] = []
            for ca, cb in cols:
                bbox = ((x1 + ca) / sx, (y1 + ra) / sy, (x1 + cb) / sx, (y1 + rb) / sy)
                txt = (page.within_bbox(bbox).extract_text() or "").replace("\n", " ")
                line.append(txt.strip())
            out.append(line)
    return out


reconstruct_table("015")


## Сравнение с regex-парсером

Смотрим, что на той же странице даёт текущий `extract_tables`.

In [ ]:
from normatives_pipeline.parser.extractors import extract_tables

for t in extract_tables(PDF):
    if t.page == 15:
        print("regex parser, page 15:", t.number, len(t.rows), "rows")
        for row in t.rows:
            print(row)


## Выводы

- **DocLayout-YOLO + TableTransformer** дают структуру таблицы (строки/колонки) по растру, без привязки к языку и к regex-подписям — это шаг к «работает для всякого PDF».
- На таблице 4.1 (стр. 15) сетка реконструируется корректно по содержимому (заголовок КС-3/КС-2/КС-1, строки уровней ответственности).
- Остаются открытые вопросы, которые надо закрыть следующими итерациями:
  1. **Разорванные по страницам таблицы** — DocLayout-YOLO видит продолжение как отдельный `table`-блок, нужна склейка/перенос номера (то же, что пришлось делать в regex-парсере).
  2. **Объединённые ячейки (spanning cells)** — TableTransformer отдаёт их как `table spanning cell`, текущая сетка их не учитывает.
  3. **Разделение header/columns** — `table column header` и `table column` надо обрабатывать отдельно, иначе заголовок дублируется в сетке.
  4. **Сканы без текстового слоя** — нужен OCR (Tesseract ru / PaddleOCR).

Следующий ноутбук: end-to-end (Nougat) и качество формул.